In [0]:
from pyspark.sql.functions import max, length, col, lit, cast

In [0]:
# Define column limits
column_limits = {
    "tvid": 250,
    "zipcode": 10,
    "dma": 128,
    "tms_episode_id": 256,
    "tivo_episode_id": 256,
    "tms_title": 256,
    "tivo_title": 256,
    "tms_airdate": 40,
    "tivo_airdate": 40,
    "tms_channel_callsign": 30,
    "tivo_channel_callsign": 30,
    "tms_channel_affiliate": 40,
    "tivo_channel_affiliate": 40,
    "is_live": 1,
    "ip_address": 32,
    "reported_input_source": 10,
    "content_type": 10,
    "tuner_content_type": 10,
    "input_category": 16,
    "input_device": 32,
    "app_service": 32,
    "tuner_tms_episode_id": 256,
    "tuner_tivo_episode_id": 256,
    "tuner_tms_title": 256,
    "tuner_tivo_title": 256,
    "tuner_tms_airdate": 40,
    "tuner_tivo_airdate": 40,
    "tuner_tms_channel_callsign": 30,
    "tuner_tivo_channel_callsign": 30,
    "tuner_tms_channel_affiliate": 40,
    "tuner_tivo_channel_affiliate": 40,
    "tuner_is_live": 1,
    "tuner_input_category": 16,
    "tuner_input_device": 32,
    "tuner_app_service": 32,
    "enableaudioacr": 1,
    "vizio_epg_channel_id": 32,
    "vizio_epg_program_id": 32,
    "tms_show_genre": 256,
    "tivo_show_genre": 256,
    "tuner_tms_show_genre": 256,
    "tuner_tivo_show_genre": 256,
    "tms_epi_title": 256,
    "tivo_epi_title": 256,
    # "tuner_tms_epi_title": 256,
    # "tuner_tivo_epi_title": 256,
    "series_id": 256,
    # "tuner_series_id": 256,
}

In [0]:
max_length_exprs = []
for col_name in column_limits:
    max_length_exprs.append(f"MAX(LENGTH(CAST({col_name} AS STRING))) AS {col_name}")

In [0]:
start_time = '2026-06-01 14:00:00'
end_time = '2026-06-02 18:00:00'

In [0]:
query = f"""
    SELECT {', '.join(max_length_exprs)}
    FROM prod.detection.viewing_content_golden
    WHERE session_start >= '{start_time}'
    AND session_start < '{end_time}'
    AND session_start_hour >= '{start_time}'
    AND session_start_hour < '{end_time}'
;"""
max_lengths_df = spark.sql(query)
max_lengths_row = max_lengths_df.collect()[0]

In [0]:
results = []
for col_name, char_limit in column_limits.items():
    actual_max = max_lengths_row[col_name]
    status = "PASS" if (actual_max is None or actual_max <= char_limit) else "FAIL"
    results.append((col_name, char_limit, actual_max, status))

In [0]:
results_df = spark.createDataFrame(results, ["column_name", "char_limit", "max_length_found", "status"])
display(results_df)

In [0]:
query = f"""
    SELECT *
    FROM dev.detection.viewing_content_golden
    WHERE session_start >= '{start_time}'
    AND session_start < '{end_time}'
    AND session_start_hour >= '{start_time}'
    AND session_start_hour < '{end_time}'
    AND LENGTH(tms_channel_affiliate) > 40
    -- GROUP BY 1
;"""
max_lengths_df = spark.sql(query).display()